# Data Preparation

In [26]:
# Imports

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix, vstack
from sklearn.feature_extraction.text import TfidfVectorizer


In [27]:
# Load Data

df = pd.read_csv('../01_data/01_raw_data/customer_original.csv')

df


,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom
...,...,...,...,...,...,...,...,...
1067366,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
1067367,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
1067368,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France
1067369,581587,22138,BAKING SET 9 PIECE RETROSPOT,3,2011-12-09 12:50:00,4.95,12680.0,France


In [28]:
df.describe()

,Quantity,Price,Customer ID
count,1.067371e+06,1.067371e+06,824364.000000
mean,9.938898e+00,4.649388e+00,15324.638504
std,1.727058e+02,1.235531e+02,1697.464450
min,-8.099500e+04,-5.359436e+04,12346.000000
25%,1.000000e+00,1.250000e+00,13975.000000
50%,3.000000e+00,2.100000e+00,15255.000000
75%,1.000000e+01,4.150000e+00,16797.000000
max,8.099500e+04,3.897000e+04,18287.000000


In [29]:
df.dtypes

Invoice            str
StockCode          str
Description        str
Quantity         int64
InvoiceDate        str
Price          float64
Customer ID    float64
Country            str
dtype: object

In [30]:
df.isna().sum()

Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

In [31]:
# Rename

df = df.rename(columns 
               = {
                   'Customer ID': 'CustomerID',
                   'Price': 'UnitPrice',
                   'Invoice': 'InvoiceNo'
}
)

In [32]:
# New Formats

df['CustomerID'] = df['CustomerID'].astype(str).fillna('Unknown')

df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"],
    format="%Y-%m-%d %H:%M:%S",
    errors="coerce"
)


## New Features

In [33]:
# New Variables (Date, Time, Revenue etc.)

# Date & Time
df['invoice_date'] = df['InvoiceDate'].dt.date
df['invoice_time'] = df['InvoiceDate'].dt.time
df['invoice_hour'] = df['InvoiceDate'].dt.hour

df['invoice_year'] = df['InvoiceDate'].dt.year
df['invoice_month'] = df['InvoiceDate'].dt.month
df['invoice_weekday_num'] = df['InvoiceDate'].dt.dayofweek  # Monday=0, Sunday=6
df['invoice_weekday'] = df['InvoiceDate'].dt.day_name()
df['is_weekend'] = df['invoice_weekday_num'].isin([5, 6])
df['invoice_calendarweek'] = df['InvoiceDate'].dt.isocalendar().week   


# Time of Day with 4 Categories
def get_time_of_day(hour):
    if 0 <= hour < 6:
        return "night"
    elif 6 <= hour < 10:
        return "morning"
    elif 10 <= hour < 14:
        return "midday"
    elif 14 <= hour < 18:
        return "afternoon"
    else:
        return "evening"

df['invoice_time_5cat'] = df['invoice_hour'].apply(get_time_of_day)


# Revenue 
df['RevenueLine'] = df['Quantity']*df['UnitPrice']

df[
    [
        'Quantity', 
        'UnitPrice',
        'RevenueLine',
    ]
].head()

df[
    [
        'InvoiceDate',
        'invoice_date',
        'invoice_calendarweek',
        'invoice_weekday',
        'is_weekend',
        'invoice_time',
        'invoice_hour',
        'invoice_time_5cat',
        'Quantity',
        'UnitPrice',
        'RevenueLine',
    ]
].head()

,InvoiceDate,invoice_date,invoice_calendarweek,invoice_weekday,is_weekend,invoice_time,invoice_hour,invoice_time_5cat,Quantity,UnitPrice,RevenueLine
0,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.95,83.4
1,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
2,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,12,6.75,81.0
3,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,48,2.10,100.8
4,2009-12-01 07:45:00,2009-12-01,49,Tuesday,False,07:45:00,7,morning,24,1.25,30.0


In [34]:
# Descriptions

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=100
)

descriptions = df["Description"].fillna("").astype(str)

X_text_line = vectorizer.fit_transform(descriptions)

customer_ids = df['CustomerID'].to_numpy()
unique_customer_ids = np.sort(np.unique(customer_ids))

customer_text_vectors = []

for customer_id in unique_customer_ids:
    customer_mask = customer_ids == customer_id
    customer_vector = X_text_line[customer_mask].mean(axis=0)

    customer_text_vectors.append(csr_matrix(customer_vector))

X_text_customer = vstack(customer_text_vectors)

In [35]:
feature_names = vectorizer.get_feature_names_out()

print(feature_names)
print(f"Number Text Features: {len(feature_names)}")

['12' '20' '60' 'antique' 'assorted' 'bag' 'bird' 'birthday' 'black'
 'blue' 'bottle' 'bowl' 'box' 'bunting' 'cake' 'cake cases' 'candle'
 'candles' 'card' 'cases' 'ceramic' 'charlotte' 'charlotte bag'
 'christmas' 'colour' 'cream' 'cutlery' 'decoration' 'design' 'dolly'
 'door' 'doormat' 'fairy' 'fairy cake' 'feltcraft' 'flower' 'frame'
 'garden' 'girl' 'glass' 'green' 'hanging' 'hanging heart' 'heart'
 'holder' 'home' 'hot' 'hot water' 'ivory' 'jumbo' 'jumbo bag' 'kit'
 'large' 'light' 'light holder' 'lights' 'love' 'lunch' 'lunch bag'
 'metal' 'metal sign' 'mini' 'mug' 'pack' 'paisley' 'paper' 'party' 'pink'
 'polkadot' 'red' 'red retrospot' 'red spotty' 'regency' 'retro'
 'retrospot' 'rose' 'set' 'sign' 'silver' 'skull' 'small' 'spaceboy'
 'spot' 'spotty' 'star' 'strawberry' 'tea' 'tin' 'trinket' 'union'
 'vintage' 'water' 'water bottle' 'white' 'wicker' 'wood' 'wooden'
 'woodland' 'wrap' 'zinc']
Number Text Features: 100


## Transactions Filtering - Purchases, Returns, Cancellations

In [36]:
# Helper variables
df["is_cancellation_invoice"] = df["InvoiceNo"].str.startswith(
    "C",
    na=False
)
df["is_return_quantity"] = df["Quantity"].lt(0)
df["is_non_positive_price"] = df["UnitPrice"].le(0)


# df Purchases
purchases = df[
    df["CustomerID"].notna()
    & df["InvoiceDate"].notna()
    & df["Description"].notna()
    & df["Quantity"].gt(0)
    & df["UnitPrice"].gt(0)
    & ~df["is_cancellation_invoice"]
].copy()

print(purchases.shape)
purchases.head()

(1041670, 22)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,invoice_date,invoice_time,...,invoice_month,invoice_weekday_num,invoice_weekday,is_weekend,invoice_calendarweek,invoice_time_5cat,RevenueLine,is_cancellation_invoice,is_return_quantity,is_non_positive_price
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,83.4,False,False,False
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,81.0,False,False,False
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,100.8,False,False,False
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,2009-12-01,07:45:00,...,12,1,Tuesday,False,49,morning,30.0,False,False,False


In [37]:
purchases = df[
    df['CustomerID'].notna()
    & df['InvoiceDate'].notna()
    & df['Description'].notna()
    & df['Quantity'].gt(0)
    & df['UnitPrice'].gt(0)
    & ~df['InvoiceNo'].astype(str).str.startswith("C", na=False)
].copy()

# purchases.describe()

In [38]:
# df Returns

returns = df[
    df['CustomerID'].notna()
    & (
        df['is_cancellation_invoice']
        | df['is_return_quantity']
    )
].copy()

returns['return_value'] = (
    returns['Quantity'] * returns['UnitPrice']
)

returns.shape

(22951, 23)

In [39]:
pd.crosstab(
    df['is_cancellation_invoice'],
    df['is_return_quantity'],
    margins=True
)

is_return_quantity,False,True,All
is_cancellation_invoice,,,
False,1044420,3457,1047877
True,1,19493,19494
All,1044421,22950,1067371


In [40]:
# df Cancellations

cancellations = df[df['UnitPrice'] < 0].copy()

cancellations[
    [
        'InvoiceNo',
        'StockCode',
        'Description',
        'Quantity',
        'UnitPrice',
        'CustomerID',
    ]
].head(20)

,InvoiceNo,StockCode,Description,Quantity,UnitPrice,CustomerID
179403,A506401,B,Adjust bad debt,1,-53594.36,Unknown
276274,A516228,B,Adjust bad debt,1,-44031.79,Unknown
403472,A528059,B,Adjust bad debt,1,-38925.87,Unknown
825444,A563186,B,Adjust bad debt,1,-11062.06,Unknown
825445,A563187,B,Adjust bad debt,1,-11062.06,Unknown


In [41]:
print("Number Customers:", purchases.groupby('CustomerID').count().shape)
print("Shape :", X_text_customer.shape)


Number Customers: (5879, 21)
Shape : (5943, 100)


In [49]:
text_feature_map = pd.DataFrame({
    "column_index": range(len(feature_names)),
    "term": feature_names
})

text_feature_map.head(50)

,column_index,term
0,0,12
1,1,20
2,2,60
3,3,antique
4,4,assorted
5,5,bag
6,6,bird
7,7,birthday
8,8,black
9,9,blue


In [48]:
# Overview Datasets

print('Purchases Dataset:', purchases.shape, '; Returns Dataset:', returns.shape, '; Cancellations Dataset:', cancellations.shape)
print('##################################################################################################')
print('Purchases Dataset:', purchases.columns)
print('##################################################################################################')
print('Returns Dataset:', returns.columns)
print('##################################################################################################')
print('Cancellations Dataset:', cancellations.columns)

Purchases Dataset: (1041670, 22) ; Returns Dataset: (22951, 23) ; Cancellations Dataset: (5, 22)
##################################################################################################
Purchases Dataset: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'invoice_date', 'invoice_time',
       'invoice_hour', 'invoice_year', 'invoice_month', 'invoice_weekday_num',
       'invoice_weekday', 'is_weekend', 'invoice_calendarweek',
       'invoice_time_5cat', 'RevenueLine', 'is_cancellation_invoice',
       'is_return_quantity', 'is_non_positive_price'],
      dtype='str')
##################################################################################################
Returns Dataset: Index(['InvoiceNo', 'StockCode', 'Description', 'Quantity', 'InvoiceDate',
       'UnitPrice', 'CustomerID', 'Country', 'invoice_date', 'invoice_time',
       'invoice_hour', 'invoice_year', 'invoice_month', 'invoice_weekday_num',

# Save Datasets with information per order

In [ ]:

## Purchases (main dataset)
purchases.to_csv('../01_data/02_processed_data/purchases_line.csv', index=False)

## Returns
returns.to_csv('../01_data/02_processed_data/returns_line.csv', index=False)

## Cancellations
cancellations.to_csv('../01_data/02_processed_data/cancellations_line.csv', index=False)

## Text data
text_feature_map.to_csv(
    "../03_results/tfidf_feature_vocabulary.csv",
    index=False
)


# Save Datasets with information on CustomerID


In [ ]:
## Purchases (main dataset)

analysis_date = purchases["InvoiceDate"].max() + pd.Timedelta(days=1)

rfm_customer = (
    purchases
    .groupby("CustomerID")
    .agg(
        recency=(
            "InvoiceDate",
            lambda dates: (analysis_date - dates.max()).days
        ),
        frequency=(
            "InvoiceNo",
            "nunique"
        ),
        monetary=(
            "RevenueLine",
            "sum"
        ),
    )
    .reset_index()
)

print(rfm_customer.shape)
rfm_customer.head()


(5879, 4)


,CustomerID,recency,frequency,monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40


In [53]:
customer_text = (
    purchases
    .groupby("CustomerID")["Description"]
    .agg(" ".join)
    .reset_index(name="customer_product_text")
)

print(customer_text.shape)
customer_text.head()

(5879, 2)


,CustomerID,customer_product_text
0,12346.0,This is a test product. This is a test product...
1,12347.0,PINK REGENCY TEACUP AND SAUCER ROSES REGENCY T...
2,12348.0,PACK OF 72 SKULL CAKE CASES 60 TEATIME FAIRY C...
3,12349.0,PLASTERS IN TIN WOODLAND ANIMALS PLASTERS IN T...
4,12350.0,CHOCOLATE THIS WAY METAL SIGN METAL SIGN NEIGH...


In [54]:
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.90,
    max_features=100
)

X_text_customer = vectorizer.fit_transform(
    customer_text["customer_product_text"]
)

customer_ids_text = customer_text["CustomerID"].to_numpy()

print(X_text_customer.shape)

(5879, 100)


In [55]:
rfm_customer = (
    rfm_customer
    .set_index("CustomerID")
    .loc[customer_ids_text]
    .reset_index()
)

assert (
    rfm_customer["CustomerID"].to_numpy() == customer_ids_text
).all()

print(rfm_customer.shape)
print(X_text_customer.shape)

(5879, 4)
(5879, 100)


In [56]:
print(rfm_customer.isna().sum())

print(
    "Duplicate customer IDs:",
    rfm_customer["CustomerID"].duplicated().sum()
)

print(
    "Customers in RFM:",
    rfm_customer["CustomerID"].nunique()
)

print(
    "Customers in text data:",
    len(customer_ids_text)
)

CustomerID    0
recency       0
frequency     0
monetary      0
dtype: int64
Duplicate customer IDs: 0
Customers in RFM: 5879
Customers in text data: 5879


In [ ]:
rfm_customer[["recency", "frequency", "monetary"]].describe()

,CustomerID,recency,frequency,monetary
0,12346.0,326,12,77556.46
1,12347.0,2,8,5633.32
2,12348.0,75,5,2019.40
3,12349.0,19,4,4428.69
4,12350.0,310,1,334.40
...,...,...,...,...
5874,18284.0,432,1,461.68
5875,18285.0,661,1,427.00
5876,18286.0,477,2,1296.43
5877,18287.0,43,7,4182.99


In [70]:
invoice_timing = (
    purchases
    .groupby(['CustomerID', 'InvoiceNo'], as_index=False)
    .agg(
        invoice_datetime=('InvoiceDate', 'min')
    )
)

invoice_timing['purchase_hour'] = (
    invoice_timing['invoice_datetime'].dt.hour
)

invoice_timing['weekday_num'] = (
    invoice_timing['invoice_datetime'].dt.dayofweek
)

invoice_timing['is_weekend'] = (
    invoice_timing['weekday_num'].isin([5, 6])
)

customer_time_profile = (
    invoice_timing
    .groupby('CustomerID')
    .agg(
        weekend_share=('is_weekend', 'mean'),
        avg_purchase_hour=('purchase_hour', 'mean'),
        preferred_hour=(
            'purchase_hour',
            lambda x: x.mode().iat[0]
        ),
        active_purchase_days=(
            'invoice_datetime',
            lambda x: x.dt.normalize().nunique()
        ),
    )
    .reset_index()
)

customer_time_profile.head()

,CustomerID,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days
0,12346.0,0.000,10.833333,13,8
1,12347.0,0.125,12.500000,14,8
2,12348.0,0.200,13.200000,10,5
3,12349.0,0.000,9.750000,9,4
4,12350.0,0.000,16.000000,16,1


In [71]:
def average_days_between_orders(dates):
    ordered_dates = pd.Series(dates).sort_values().drop_duplicates()

    if len(ordered_dates) < 2:
        return np.nan

    return ordered_dates.diff().dropna().dt.total_seconds().div(
        86_400
    ).mean()

purchase_rhythm = (
    invoice_timing
    .groupby('CustomerID')['invoice_datetime']
    .agg(
        avg_days_between_orders=average_days_between_orders,
        order_date_count="nunique"
    )
    .reset_index()
)

purchase_rhythm.head()

,CustomerID,avg_days_between_orders,order_date_count
0,12346.0,36.369129,12
1,12347.0,57.437698,8
2,12348.0,90.731597,5
3,12349.0,190.284954,4
4,12350.0,NaN,1


In [81]:
customer_profile = (
    rfm_customer
    .merge(customer_time_profile, on='CustomerID', how='left')
    .merge(purchase_rhythm, on="CustomerID", how='left')
    .merge(customer_text, on='CustomerID', how='left'
))

customer_profile

,CustomerID,recency,frequency,monetary,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,avg_days_between_orders,order_date_count,customer_product_text
0,12346.0,326,12,77556.46,0.000000,10.833333,13,8,36.369129,12,This is a test product. This is a test product...
1,12347.0,2,8,5633.32,0.125000,12.500000,14,8,57.437698,8,PINK REGENCY TEACUP AND SAUCER ROSES REGENCY T...
2,12348.0,75,5,2019.40,0.200000,13.200000,10,5,90.731597,5,PACK OF 72 SKULL CAKE CASES 60 TEATIME FAIRY C...
3,12349.0,19,4,4428.69,0.000000,9.750000,9,4,190.284954,4,PLASTERS IN TIN WOODLAND ANIMALS PLASTERS IN T...
4,12350.0,310,1,334.40,0.000000,16.000000,16,1,NaN,1,CHOCOLATE THIS WAY METAL SIGN METAL SIGN NEIGH...
...,...,...,...,...,...,...,...,...,...,...,...
5874,18284.0,432,1,461.68,0.000000,11.000000,11,1,NaN,1,CARRIAGE CARDHOLDER GINGHAM CHRISTMAS TREE GRE...
5875,18285.0,661,1,427.00,0.000000,10.000000,10,1,NaN,1,GLASS CAKE STAND MIRRORED BASE CAKE STAND VICT...
5876,18286.0,477,2,1296.43,0.000000,10.500000,10,2,247.050000,2,S/4 ROSE PINK DINNER CANDLES VICTORIAN GLASS H...
5877,18287.0,43,7,4182.99,0.142857,10.714286,10,6,88.149769,7,"HOOK, 5 HANGER ,MAGIC TOADSTOOL RED GARLAND, M..."


In [80]:
customer_profile.isna().sum()

CustomerID                    0
recency                       0
frequency                     0
monetary                      0
weekend_share                 0
avg_purchase_hour             0
preferred_hour                0
active_purchase_days          0
avg_days_between_orders    1625
order_date_count              0
customer_product_text         0
dtype: int64

In [79]:
customer_profile.describe()

,recency,frequency,monetary,weekend_share,avg_purchase_hour,preferred_hour,active_purchase_days,avg_days_between_orders,order_date_count
count,5879.000000,5879.000000,5.879000e+03,5879.000000,5879.000000,5879.000000,5879.000000,4254.000000,5879.000000
mean,201.297840,6.816976,3.567374e+03,0.139721,12.641307,11.963769,5.724783,102.559841,6.746726
std,209.337205,42.492965,4.458180e+04,0.263779,1.743371,2.240601,12.255591,98.454007,39.586700
min,1.000000,1.000000,2.950000e+00,0.000000,7.000000,6.000000,1.000000,0.000694,1.000000
25%,26.000000,1.000000,3.487750e+02,0.000000,11.633971,10.000000,1.000000,38.777908,1.000000
50%,96.000000,3.000000,8.989600e+02,0.000000,12.500000,12.000000,3.000000,72.566937,3.000000
75%,380.000000,7.000000,2.309050e+03,0.166667,13.666667,14.000000,6.000000,129.987847,7.000000
max,739.000000,3108.000000,3.229165e+06,1.000000,20.000000,20.000000,549.000000,714.152083,2876.000000


In [76]:
## Purchases (main dataset)
customer_profile.to_csv('../01_data/02_processed_data/customer_profile.csv', index=False)
